# Tutorial 2 &mdash; Tissue niches: what sits next to what, and why it matters

**ASI-FIMSA Workshop 2026 &mdash; spatial omics, hands-on**

A cell type list tells you *who is in the room*. It does not tell you *who is standing with
whom* &mdash; and in a tumour, that is most of the biology. A T cell wedged between two
carcinoma cells is doing something very different from a T cell sitting in collagen 300 µm
away, even though both are labelled "T cell" in the table.

This Tutorial is about turning that intuition into numbers. We work on a 2,000 µm square of
an invasive breast carcinoma, imaged with **Atera** whole-transcriptome Xenium &mdash;
16,006 individual cells, each with a position, a boundary polygon and a cell type &mdash; and
we ask the same question three times, each time more sharply:

| Section | The question | What you get back |
| --- | --- | --- |
| **3** | Which cell **types** sit together more often than chance? | one number per pair of types |
| **4** | What **niches** does this tissue actually contain? | a niche label **for every cell** |
| **5** | How **big**, how **round**, how **far apart** are those niches? | niches as measurable objects |

**What you will get out of it**

1. A working understanding of **neighbourhood composition** &mdash; the one idea underneath
   almost every spatial niche method in the literature.
2. Niches you built yourself, in about fifteen lines, named from their own composition:
   tumour core, tumour boundary, fibroblast stroma, **immune infiltrate**.
3. A per-cell measurement of **how far each cell sits from the immune niche**, which is the
   quantitative version of "immune-excluded tumour".

Everything runs on a free Colab CPU session in a few minutes. There is no GPU anywhere in
this Tutorial.

> The kNN + KMeans method in Section 4 is adapted from `3.2_neighborhood.ipynb` in the
> `gml-teaching-2026` course. Its R sibling, `3.1_hoodscanR.ipynb`, does the same job with
> [**hoodscanR**](https://bioconductor.org/packages/hoodscanR/); this Workshop is Python-only,
> so hoodscanR appears in the further reading at the end rather than in a cell.

## 0. Setup

### 0.1 Install

Colab already ships **numpy**, **pandas**, **matplotlib**, **scikit-learn**, **scipy** and
**seaborn**. We add four things: **squidpy** (spatial statistics), **sopa** (spatial-omics
pipeline utilities, which also pulls in **spatialdata** for reading the Crop), **scanpy**
(the AnnData ecosystem underneath both) and a current **seaborn**.

This takes two to three minutes. Run it, then read Section 0.2 while it works.

> **Why the pin file.** The Workshop pins every version in `constraints-colab.txt`, and the
> cell below fetches it before installing. The version that matters most is **numpy 2.0.2**,
> which Colab has already loaded into memory before your first cell runs. If an install
> silently upgrades numpy, Colab demands a session restart &mdash; halfway through a
> Tutorial, for a whole room at once. If the pin file cannot be fetched we install anyway
> and only the pinning is lost.

In [ ]:
# ============================================================================
# Install. You do not need to read or understand this cell -- just run it.
# ============================================================================
import subprocess
import sys
import urllib.request

IN_COLAB = "google.colab" in sys.modules

REPO_RAW = "https://raw.githubusercontent.com/xiao233333/ASI-FIMSA-workshop-2026/main"
PACKAGES = ["squidpy", "sopa", "scanpy", "seaborn"]

if IN_COLAB:
    args = list(PACKAGES)
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/constraints-colab.txt",
                                   "constraints-colab.txt")
        args = ["-c", "constraints-colab.txt"] + args
        print("using the Workshop pin set (constraints-colab.txt)")
    except Exception as exc:                       # noqa: BLE001
        print(f"could not fetch the pin set ({type(exc).__name__}); "
              "installing unpinned, which is usually fine")
    print("installing:", " ".join(PACKAGES), "... this takes 2-3 minutes")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)
    print("done")
else:
    print("Not running in Google Colab -- assuming squidpy, sopa, scanpy and")
    print("seaborn are already installed in this environment.")

### 0.2 Imports, and one small helper we download

Everything below is a normal import except the last one. `voronoi.py` is a small plotting
helper vendored from the parent course, used once in Section 4. Colab opens a notebook on its
own, without the files that sit next to it in the repository, so we fetch it.

If the fetch fails, nothing else breaks &mdash; one figure in Section 4 is skipped and the
Tutorial carries on.

In [ ]:
import os
import sys
import time
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spatialdata as sd
import squidpy as sq
import sopa
import sopa.spatial
from matplotlib.collections import LineCollection
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
SEED = 0
np.random.seed(SEED)

print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("spatialdata :", sd.__version__)
print("squidpy     :", sq.__version__)
print("sopa        :", sopa.__version__)

# --- the vendored Voronoi helper -------------------------------------------
REPO_RAW = "https://raw.githubusercontent.com/xiao233333/ASI-FIMSA-workshop-2026/main"
HF_REPO = "xiao233333/asi-fimsa-workshop-2026"
HAVE_VORONOI = False

if not Path("voronoi.py").exists():
    import urllib.request
    for source in ("github", "huggingface"):
        try:
            if source == "github":
                urllib.request.urlretrieve(f"{REPO_RAW}/notebooks/voronoi.py", "voronoi.py")
            else:
                from huggingface_hub import hf_hub_download
                Path("voronoi.py").write_text(Path(hf_hub_download(
                    repo_id=HF_REPO, filename="voronoi.py", repo_type="dataset")).read_text())
            break
        except Exception:                          # noqa: BLE001, S112
            continue

try:
    sys.path.insert(0, ".")
    from voronoi import draw_voronoi_scatter
    HAVE_VORONOI = True
    print("voronoi.py  : ready")
except Exception as exc:                           # noqa: BLE001
    print(f"voronoi.py  : not available ({type(exc).__name__}) -- "
          "one optional figure in Section 4 will be skipped")

### 0.3 Download the Crop

We work on one prepared file, **`atera_crop.zarr.zip`** (18 MB). It holds a 2,000 µm square
of the slide as a `SpatialData` object: an H&E overview, a full-resolution zoom, 16,006 cell
boundary polygons, and a table of 16,006 cells &times; 69 genes with a cell type per cell.

Two practical notes, both of which will bite you on your own data one day:

* **A `.zarr.zip` has to be unzipped before it can be read.** `spatialdata.read_zarr()` needs
  a directory, not a zip archive &mdash; on spatialdata 0.7.x it resolves the store to a
  local path and asks for its root, which a zip store does not have. The cell below unzips
  first. (The file itself says so: look at `sdata.attrs["atera"]["read_instructions"]`.)
* **Zarr is a directory of many small files**, which is why it is shipped zipped. One 18 MB
  download beats 183 separate ones.

The cell tries Hugging Face first, then Google Drive, and honours a `WORKSHOP_DATA_DIR`
environment variable if you already have the file locally.

In [ ]:
# ============================================================================
# Get atera_crop.zarr.zip, then unzip it to atera_crop.zarr/
# ============================================================================
HF_REPO = "xiao233333/asi-fimsa-workshop-2026"
CROP_ZIP_NAME = "atera_crop.zarr.zip"
CROP_DIR = Path("atera_crop.zarr")
# Google Drive mirror; the presenter sets this if Hugging Face is unreachable.
DRIVE_FILE_ID = os.environ.get("WORKSHOP_CROP_DRIVE_ID", "")

crop_zip = None

# 1. already downloaded, or pointed at a local copy
for candidate in [Path(os.environ.get("WORKSHOP_DATA_DIR", ".")) / CROP_ZIP_NAME,
                  Path(CROP_ZIP_NAME)]:
    if candidate.exists():
        crop_zip = candidate
        print(f"using local copy: {crop_zip}")
        break

# 2. Hugging Face (primary)
if crop_zip is None:
    try:
        from huggingface_hub import hf_hub_download
        crop_zip = Path(hf_hub_download(repo_id=HF_REPO, filename=CROP_ZIP_NAME,
                                        repo_type="dataset"))
        print(f"downloaded from Hugging Face: {HF_REPO}")
    except Exception as exc:                       # noqa: BLE001
        # Deliberately no traceback -- a wall of red text reads like something
        # you did wrong, and this is expected until the dataset is published.
        print(f"Hugging Face is not reachable yet ({type(exc).__name__}).")

# 3. Google Drive (mirror)
if crop_zip is None and DRIVE_FILE_ID:
    try:
        import gdown
        gdown.download(id=DRIVE_FILE_ID, output=CROP_ZIP_NAME, quiet=True)
        crop_zip = Path(CROP_ZIP_NAME)
        assert crop_zip.exists()
        print("downloaded from the Google Drive mirror")
    except Exception as exc:                       # noqa: BLE001
        crop_zip = None
        print(f"the Google Drive mirror is not reachable either "
              f"({type(exc).__name__}).")

if crop_zip is None:
    print()
    print("Could not fetch the Crop. Please tell the presenter -- this is our")
    print("problem, not yours. If you have the file already, put it next to this")
    print("notebook, or set WORKSHOP_DATA_DIR to the folder holding it, and")
    print("re-run this cell.")
else:
    print(f"{crop_zip.name}: {crop_zip.stat().st_size / 1e6:.1f} MB")
    if not CROP_DIR.exists():
        t0 = time.time()
        with zipfile.ZipFile(crop_zip) as zf:
            zf.extractall(CROP_DIR)
        print(f"unzipped to {CROP_DIR}/ in {time.time() - t0:.1f} s")
    else:
        print(f"{CROP_DIR}/ already unzipped")

## 1. The Crop

`read_zarr` gives back a `SpatialData` object: several *elements* &mdash; images, shapes,
tables &mdash; that all live in the same coordinate system, so anything you measure on one
lines up with the others. Here that coordinate system is called `"global"` and its units are
**micrometres**, which is why every distance in this Tutorial is a real distance and not a
pixel count.

In [ ]:
sdata = sd.read_zarr(CROP_DIR)
print(sdata)

In [ ]:
adata = sdata["table"]        # the cell table, an ordinary AnnData
print(adata)
print()
print("coordinates (obsm['spatial']), in micrometres:")
print("  x:", adata.obsm["spatial"][:, 0].min().round(0), "to",
      adata.obsm["spatial"][:, 0].max().round(0))
print("  y:", adata.obsm["spatial"][:, 1].min().round(0), "to",
      adata.obsm["spatial"][:, 1].max().round(0))
print()
print("cell types:")
print(adata.obs["cell_type"].value_counts().to_string())

### 1.1 Why a 2,000 µm square and not the whole slide?

Because the whole slide does not fit, and being honest about that is part of the method.

| | whole slide | this Crop |
| --- | --- | --- |
| cells | 170,057 | **16,006** |
| genes measured | 18,028 (whole transcriptome) | 69 (a marker panel carried along) |
| non-zero counts | ~307 million | ~0.4 million |
| count matrix in memory | **~2.5 GB** | ~4 MB |
| H&E + boundaries | tens of GB | 18 MB |

A free Colab session gives you roughly 12 GB of RAM, shared with everything else you have
loaded. A 2.5 GB dense matrix is not fatal on its own, but by the time you have a neighbour
graph, a copy for scaling and a plotting buffer alongside it, you are restarting the session.

So we cropped &mdash; and **how** we cropped matters. The window was chosen by maximum cell
count *subject to a floor on the diversity of its cell-type composition*. The densest 2,000 µm
window on this slide holds 20,779 cells, but it is essentially solid tumour, and a niche
analysis of solid tumour finds one niche. Trading ~5,000 cells for a genuine tumour / stroma /
immune mix is the entire reason this Tutorial has something to show you.

> **The general lesson.** "Subset until it fits" is a modelling decision, not a technicality.
> Say out loud what you optimised the subset for, because that is what your niches will be
> made of.

In [ ]:
# What the Crop is, in its own words.
meta = sdata.attrs["atera"]
print(meta["dataset"])
print("chemistry :", meta["chemistry_version"])
print("frame     :", meta["frame"])
print("units     :", meta["units"])
print()
print("read me   :", meta["read_instructions"])

### 1.2 Look at it

Two panels. On the left the H&E, which is what a pathologist would look at. On the right the
same square with one dot per cell, coloured by cell type.

Spend a moment matching them up. The dense pink-purple islands on the H&E are the carcinoma
nests, and they come out solid red on the right. The paler fibrillar material between them is
collagenous stroma, and that is where the blue and cyan dots &mdash; T cells, dendritic cells,
macrophages &mdash; live.

**That visual impression is the whole Tutorial.** Everything from here is an attempt to
measure it.

In [ ]:
from spatialdata.transformations import get_transformation


def image_and_extent(sdata, name):
    """Return an image element as an (H, W, 3) RGB array plus a matplotlib extent.

    The extent is in micrometres, taken from the element's own transformation to the
    "global" coordinate system, so the image and the cell coordinates overlay correctly
    without anyone hard-coding a pixel size.
    """
    el = sdata[name]
    rgb = np.asarray(el.transpose("y", "x", "c").data)
    M = get_transformation(el, "global").to_affine_matrix(
        input_axes=("x", "y"), output_axes=("x", "y"))
    h, w = rgb.shape[:2]
    (x0, x1), (y0, y1) = (M @ np.array([[0, 0, 1], [w, h, 1]]).T)[:2]
    return rgb, (x0, x1, y1, y0)          # y flipped: images draw top-down


he, HE_EXTENT = image_and_extent(sdata, "he")
print("H&E overview:", he.shape, " extent (µm):",
      [round(float(v)) for v in HE_EXTENT])

CELL_TYPES = list(adata.obs["cell_type"].cat.categories)
PALETTE = dict(zip(CELL_TYPES, adata.uns["cell_type_colors"]))
xy = adata.obsm["spatial"]

fig, axes = plt.subplots(1, 2, figsize=(13, 6.4))
axes[0].imshow(he, extent=HE_EXTENT)
axes[0].set_title("H&E")
axes[1].imshow(he, extent=HE_EXTENT, alpha=0.30)
for ct in CELL_TYPES:
    m = (adata.obs["cell_type"] == ct).values
    axes[1].scatter(xy[m, 0], xy[m, 1], s=1.8, c=PALETTE[ct], linewidths=0,
                    label=f"{ct} ({m.sum():,})")
axes[1].set_title(f"{adata.n_obs:,} cells, coloured by type")
axes[1].legend(markerscale=6, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
for ax in axes:
    ax.set_xlim(HE_EXTENT[0], HE_EXTENT[1])
    ax.set_ylim(HE_EXTENT[2], HE_EXTENT[3])
    ax.set_aspect("equal")
    ax.set_xlabel("x (µm)")
axes[0].set_ylabel("y (µm)")
plt.tight_layout()
plt.show()

In [ ]:
# Composition of the Crop, as a bar chart.
counts = adata.obs["cell_type"].value_counts()
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.barh(counts.index[::-1], counts.values[::-1],
        color=[PALETTE[c] for c in counts.index[::-1]])
for i, (ct, n) in enumerate(zip(counts.index[::-1], counts.values[::-1])):
    ax.text(n + 60, i, f"{n:,}  ({100 * n / adata.n_obs:.1f}%)",
            va="center", fontsize=8)
ax.set_xlim(0, counts.max() * 1.30)
ax.set_xlabel("cells")
ax.set_title(f"Cell-type composition of the Crop  (n = {adata.n_obs:,})")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

Two things to notice before we go on.

**It is a tumour-dominated tissue.** 43% of the cells are tumour epithelium and another 7%
are proliferating tumour. Any statistic that does not correct for that will simply rediscover
"most cells are tumour cells" &mdash; which is why Section 3 compares against a shuffled null
rather than reporting raw counts.

**400 cells are `Unassigned`, and they stay in.** These are cells whose vendor cluster showed
no marker evidence strong enough to name. Dropping them would quietly delete the cells the
method understood least, which is exactly the wrong direction. They are 2.5% of the Crop and
they carry no weight in the story; leaving them visible is the honest option.

### 1.3 The same tissue at cellular resolution

The Crop also carries `he_zoom`: a 300 µm sub-square at the **full** H&E resolution
(0.27 µm per pixel), with the per-cell boundary polygons drawn over it.

This panel is worth dwelling on, because it is where the segmentation stops being an
abstraction. Every coloured outline is one cell that the instrument called, and every dot in
every other figure in this Tutorial is the centroid of one of these.

In [ ]:
he_zoom, ZOOM_EXTENT = image_and_extent(sdata, "he_zoom")
zx0, zx1, zy1, zy0 = ZOOM_EXTENT

boundaries = sdata["cell_boundaries"]
sub = boundaries.cx[zx0:zx1, zy0:zy1].copy()
sub["cell_type"] = adata.obs["cell_type"].reindex(sub.index).values
print(f"{len(sub)} cells in the {zx1 - zx0:.0f} µm zoom window")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.4))
axes[0].imshow(he_zoom, extent=ZOOM_EXTENT)
axes[0].set_title("H&E at full resolution (0.27 µm/pixel)")
axes[1].imshow(he_zoom, extent=ZOOM_EXTENT)
sub.plot(ax=axes[1], color=[PALETTE[c] for c in sub["cell_type"]],
         edgecolor="black", linewidth=0.25, alpha=0.55)
axes[1].set_title("the same field, with segmented cells")
for ax in axes:
    ax.set_xlim(zx0, zx1)
    ax.set_ylim(zy1, zy0)
    ax.set_aspect("equal")
    ax.axis("off")
plt.tight_layout()
plt.show()

Look at the middle band: a stream of small round blue cells (T cells) running between dense
pink collagen above and the edge of a carcinoma duct below. On the H&E alone you would call
that "lymphocytic infiltrate at the tumour margin" and move on. By the end of Section 4 that
band will have its own label, its own composition, and a measurable distance to the tumour.

### 1.4 An honest aside: where are the B cells?

The cell types here come from unsupervised clustering of the vendor's own graph clusters,
mapped onto marker panels. **There is no B cell cluster.** Not because B cells were excluded,
but because the clustering did not resolve one.

All four canonical B-cell genes &mdash; `MS4A1` (CD20), `CD79A`, `CD79B`, `BANK1` &mdash; are
in the 69-gene panel shipped with this Crop, so you can interrogate the question yourself.
Run the next cell before you read the discussion under it.

In [ ]:
B_GENES = ["MS4A1", "CD79A", "CD79B", "BANK1"]
X = np.asarray(adata.X)                       # raw counts, dense, (cells x 69 genes)
gene_index = {g: i for i, g in enumerate(adata.var_names)}
print("all four B-cell genes in the panel:", all(g in gene_index for g in B_GENES))

cols = [gene_index[g] for g in B_GENES]
any_b = (X[:, cols] > 0).any(axis=1)
print(f"\ncells with at least one count of ANY of {B_GENES}: "
      f"{any_b.sum():,} / {adata.n_obs:,}  ({100 * any_b.mean():.2f}%)")
ms4a1_pos = X[:, gene_index["MS4A1"]] > 0
print(f"cells with at least one MS4A1 count:              "
      f"{ms4a1_pos.sum():,} / {adata.n_obs:,}  ({100 * ms4a1_pos.mean():.2f}%)")

tbl = pd.DataFrame({
    "n cells": adata.obs["cell_type"].value_counts(),
    "% any B gene": (pd.Series(any_b, index=adata.obs_names)
                     .groupby(adata.obs["cell_type"], observed=True).mean() * 100).round(1),
    "% MS4A1+": (pd.Series(ms4a1_pos, index=adata.obs_names)
                 .groupby(adata.obs["cell_type"], observed=True).mean() * 100).round(1),
    "mean MS4A1 counts": (pd.Series(X[:, gene_index["MS4A1"]], index=adata.obs_names)
                          .groupby(adata.obs["cell_type"], observed=True).mean()).round(2),
})
print()
print(tbl.sort_values("% MS4A1+", ascending=False).to_string())

Read that table carefully, because it contains two different lessons.

**One count is not a cell type.** About 10% of cells carry at least one count of *some*
B-cell gene &mdash; including 11% of tumour epithelial cells, which certainly are not B cells.
That number is mostly ambient transcript and segmentation bleed, and it is why "% of cells
positive for gene X" is a treacherous statistic in imaging-based spatial transcriptomics.
The specific column is `MS4A1`: ~3% of cells, and its highest mean is in the **T cell**
cluster.

**B cells here are real but unresolved.** They are genuinely uncommon in this tumour, and the
few that exist sit inside the T-cell cluster rather than pulling away into their own. The
Crop's cluster annotation was reviewed by hand and no B cell label was forced, because a
cluster should not be handed a name it has not earned.

> **Why we are telling you this rather than hiding it.** Unsupervised clustering resolves
> what is abundant and distinct. A rare population with a subtle profile hides inside its
> nearest neighbour, and no amount of re-running Leiden fixes that. If B cells were the
> question you came with, this dataset would answer it by *targeted* means &mdash; score every
> cell for the four genes and look at where the high scorers sit &mdash; not by clustering.
> Exercise 4 asks you to do exactly that, and the niches from Section 4 are what you would
> score them against.

---

## 2. What is a niche?

Take one cell. Draw a small circle around it. Write down what fraction of the cells inside
that circle are tumour, what fraction are T cells, what fraction are fibroblasts. That list of
fractions is the cell's **neighbourhood composition** &mdash; a short vector that describes
the cell's *surroundings* rather than the cell itself.

Now do that for all 16,006 cells and cluster the vectors. Cells whose surroundings look alike
land in the same cluster, and each cluster is a **niche**: a recurring local arrangement of
cell types. Not a place &mdash; a *kind of place*. The same niche can appear in twenty
separate spots across the tissue, which is precisely what makes it worth naming.

Three details decide what you get, and all three are choices you make, not facts you discover:

| Choice | What it controls | What we use |
| --- | --- | --- |
| how you define "nearby" | the **spatial scale** of a niche | the 20 nearest cells |
| what you count | what the niches are made **of** | cell types |
| how many clusters | how **finely** the tissue is divided | 8 |

Every published niche method is a variation on this. `hoodscanR` uses a soft probabilistic
assignment instead of hard KMeans; CellCharter smooths the neighbourhood vector over several
hops and fits a Gaussian mixture; UTAG multiplies the expression matrix by the adjacency
matrix before clustering. The skeleton is the same one you are about to write.

### 2.1 Make it concrete first

Before we touch 16,006 real cells, here is the whole idea on 900 fake ones. A round nest of
"Tumour" cells, a ring of "Immune" cells around it, "Stroma" everywhere else. We take two
**tumour** cells &mdash; one buried in the middle of the nest, one on its outer rim &mdash;
and compute the composition of their 10 nearest neighbours.

Same cell type, completely different neighbourhoods. That difference is what we are about to
cluster on, and the whole of Section 4 is this cell run on real data.

In [ ]:
# ---- a toy tissue: 900 cells on a jittered grid --------------------------
rng = np.random.default_rng(SEED)
gx, gy = np.meshgrid(np.arange(30.0), np.arange(30.0))
demo_xy = np.c_[gx.ravel(), gy.ravel()] + rng.normal(0, 0.15, (900, 2))

r = np.hypot(demo_xy[:, 0] - 14.5, demo_xy[:, 1] - 14.5)
demo_type = np.where(r < 7, "Tumour", np.where(r < 10, "Immune", "Stroma"))

# ---- the whole method, in five lines -------------------------------------
onehot = pd.get_dummies(demo_type).astype(float)          # one column per cell type
nn = NearestNeighbors(n_neighbors=10).fit(demo_xy)        # "nearby" = 10 nearest cells
_, idx = nn.kneighbors(demo_xy)                           # idx[i] = neighbours of cell i
windows = onehot.values[idx].sum(axis=1) / 10             # composition, one row per cell
demo_comp = pd.DataFrame(windows, columns=onehot.columns)

# ---- two cells, picked by hand -------------------------------------------
is_tumour = demo_type == "Tumour"
cell_A = int(np.argmin(r))                                # deepest inside the nest
cell_B = int(np.argmax(np.where(is_tumour, r, -1)))       # tumour cell on the rim
for label, cell in [("cell A", cell_A), ("cell B", cell_B)]:
    print(f"{label}: a {demo_type[cell]} cell, {r[cell]:.1f} units from the nest centre")
print()
print(pd.DataFrame({"cell A (nest centre)": demo_comp.iloc[cell_A],
                    "cell B (tumour edge)": demo_comp.iloc[cell_B]}).to_string())

In [ ]:
DEMO_COLOURS = {"Tumour": "#c1272d", "Immune": "#1f6fb4", "Stroma": "#8c6d3f"}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.6))
for ax, cell in zip(axes, [cell_A, cell_B]):
    for t, c in DEMO_COLOURS.items():
        m = demo_type == t
        ax.scatter(demo_xy[m, 0], demo_xy[m, 1], s=16, c=c, label=t, linewidths=0)
    nb = idx[cell]
    ax.scatter(demo_xy[nb, 0], demo_xy[nb, 1], s=110, facecolors="none",
               edgecolors="black", linewidths=1.1)
    ax.scatter(*demo_xy[cell], s=180, marker="*", c="yellow",
               edgecolors="black", linewidths=0.8, zorder=5)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
axes[0].set_title("cell A: a tumour cell whose neighbourhood is all tumour")
axes[1].set_title("cell B: a tumour cell at the immune interface")
axes[0].legend(loc="upper left", fontsize=8, markerscale=1.4)
plt.suptitle("Same idea, 900 fake cells: star = the cell, circles = its 10 nearest neighbours")
plt.tight_layout()
plt.show()

Cell A and cell B are both tumour cells. Nothing about their own transcriptome distinguishes
them in this toy example. But their **neighbourhood vectors** are completely different, and
in real tissue that difference is a real biological difference: cell B is at an interface
where immune contact, hypoxia gradients and matrix remodelling all happen, and cell A is not.

That is all a niche method does. The rest is bookkeeping.

---

## 3. squidpy: which cell types sit together?

The quickest question to ask of real data is the pairwise one: **do T cells sit next to tumour
cells more often, or less often, than they would if the labels were shuffled?**

`squidpy` answers it in two steps.

**Step 1 &mdash; build a neighbour graph.** Decide, once, which cells count as neighbours of
which. We use a **Delaunay triangulation**: connect two cells if their Voronoi territories
share an edge. It is the natural "who touches whom" graph for a tissue &mdash; it needs no
distance threshold, it adapts automatically to dense and sparse regions, and it is what
`sopa` will require later in Section 5.

**Step 2 &mdash; permutation test.** Count how many graph edges join each pair of cell types.
Then shuffle the cell type labels across the cells (keeping the graph fixed), recount, and
repeat 200 times. The result is a **z-score** per pair: how many standard deviations the real
count sits above or below the shuffled ones. Positive means "together more often than
chance", negative means "kept apart".

> **The first `nhood_enrichment` call is slow.** squidpy compiles the permutation loop with
> `numba` the first time it runs, which can take **up to 30 seconds** before any work starts.
> That is compilation, not computation. Run it once and it stays fast.

In [ ]:
t0 = time.time()
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)
print(f"Delaunay graph built in {time.time() - t0:.1f} s")

A = adata.obsp["spatial_connectivities"]
D = adata.obsp["spatial_distances"]
print(f"  {A.nnz // 2:,} edges for {adata.n_obs:,} cells "
      f"(mean {A.nnz / adata.n_obs:.1f} neighbours per cell)")
print(f"  edge length: median {np.median(D.data):.1f} µm, "
      f"95th percentile {np.percentile(D.data, 95):.1f} µm, "
      f"longest {D.data.max():.0f} µm")

Note that longest edge. A Delaunay triangulation has no upper bound on edge length, so cells
on opposite sides of an empty duct lumen &mdash; or on the convex hull of the tissue &mdash;
get joined by edges hundreds of micrometres long. They are a small minority (95% of edges are
under ~36 µm, roughly one cell diameter) and they do not distort the enrichment test, but in
Section 5 we will prune them explicitly, because a *distance* computed along a graph with
1,700 µm shortcuts in it would be nonsense.

### 3.1 See the graph before you trust it

Statistics computed on a graph are only as good as the graph. Here it is, drawn over a 500 µm
sub-square, with cells coloured by type.

In [ ]:
SUB_X0, SUB_Y0, SUB_SIZE = 4300.0, 10400.0, 500.0        # a 500 µm window, ~1,300 cells
in_window = ((xy[:, 0] >= SUB_X0) & (xy[:, 0] < SUB_X0 + SUB_SIZE) &
             (xy[:, 1] >= SUB_Y0) & (xy[:, 1] < SUB_Y0 + SUB_SIZE))
print(f"{in_window.sum():,} cells in the sub-crop")

coo = A.tocoo()
keep = in_window[coo.row] & in_window[coo.col] & (coo.row < coo.col)
segments = np.stack([xy[coo.row[keep]], xy[coo.col[keep]]], axis=1)

fig, ax = plt.subplots(figsize=(7.2, 7))
ax.add_collection(LineCollection(segments, colors="0.75", linewidths=0.4, zorder=1))
for ct in CELL_TYPES:
    m = in_window & (adata.obs["cell_type"] == ct).values
    if m.sum():
        ax.scatter(xy[m, 0], xy[m, 1], s=11, c=PALETTE[ct], label=ct, linewidths=0, zorder=2)
ax.set_xlim(SUB_X0, SUB_X0 + SUB_SIZE)
ax.set_ylim(SUB_Y0 + SUB_SIZE, SUB_Y0)
ax.set_aspect("equal")
ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm)")
ax.set_title(f"Delaunay neighbour graph, {len(segments):,} edges in a 500 µm window")
ax.legend(fontsize=6.5, markerscale=1.6, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

This is the object the next cell does statistics on. You can see a duct in section &mdash;
red tumour cells packed inside a rim of orange myoepithelial cells &mdash; with blue T cells
and cyan dendritic cells outside it in the stroma. Already, by eye, the tumour and the
lymphocytes are barely connected to each other. Now measure it.

In [ ]:
t0 = time.time()
sq.gr.nhood_enrichment(adata, cluster_key="cell_type", n_perms=200, seed=SEED,
                       show_progress_bar=False)
print(f"nhood_enrichment: {time.time() - t0:.1f} s  (200 permutations)")

fig, ax = plt.subplots(figsize=(7.6, 6.4))
sq.pl.nhood_enrichment(adata, cluster_key="cell_type", cmap="coolwarm",
                       vmin=-60, vmax=60, ax=ax,
                       title="Neighbourhood enrichment (z-score vs shuffled labels)")
plt.tight_layout()
plt.show()

In [ ]:
# The same numbers, as a table, so we can quote them.
z = pd.DataFrame(adata.uns["cell_type_nhood_enrichment"]["zscore"],
                 index=CELL_TYPES, columns=CELL_TYPES)

print("What tumour epithelium sits next to (z-score, most to least):")
print(z.loc["Tumour epithelial"].sort_values(ascending=False).round(1).to_string())
print()
print("What T cells sit next to:")
print(z.loc["T cell"].sort_values(ascending=False).round(1).to_string())

### 3.2 Read it as immunology

The heatmap has a very clear block structure, and every block is a statement about this
tumour.

**The tumour compartment is closed.** `Tumour epithelial` is enormously self-enriched
(z &asymp; +119) and **negative against every immune population**: T cells &asymp; &minus;72,
dendritic cells &asymp; &minus;43, macrophages &asymp; &minus;34. `Proliferating tumour` sits
with it. This is the signature of an **immune-excluded** tumour: the lymphocytes are present
in the tissue, in decent numbers, and they are not getting in.

**The immune compartment is one warm block.** T cells, dendritic cells, macrophages, plasma
cells and mast cells are all positively enriched with each other *and* with fibroblasts. The
strongest immune pair after self-association is T cell with the mixed plasma/mast population
(&asymp; +40) and T cell with dendritic cell (&asymp; +29) &mdash; T cells and antigen-presenting
cells co-localising in stroma, which is what you would hope to see.

**Vasculature is the positive control.** `Endothelial` and `Perivascular` sit at &asymp; +41
with each other &mdash; pericytes wrapping endothelium. Nobody needed an algorithm to predict
that, which is exactly why it is reassuring: a method that missed it would be broken.

**Myoepithelium marks the wall.** `Myoepithelial` is self-enriched (&asymp; +51), mildly
positive with proliferating tumour, and *negative* with T cells (&asymp; &minus;25). It is the
physical layer between the ducts and the stroma, and it shows up here as one.

### 3.3 The limitation that motivates the rest of the Tutorial

Look at what you have: a 13 &times; 13 matrix. One number per pair of cell types, **for the
whole tissue at once**.

That is a global statistic. It cannot tell you:

* **Where.** Is the exclusion uniform, or is there one duct that lymphocytes have breached?
* **Which cells.** Which specific T cells are the infiltrating ones?
* **How much tissue.** What fraction of this Crop is "immune-rich stroma" as opposed to
  "tumour with a thin immune rim"?

You cannot answer any of those with a number per pair, because a number per pair has no
per-cell resolution at all. For that, every cell needs its own label.

That is Section 4.

---

## 4. Build niches yourself

This is the heart of the Tutorial, and it is genuinely short. Four steps:

```
1.  one-hot encode the cell types          ->  a 0/1 matrix, cells x types
2.  find the k nearest cells to each cell  ->  a k-column index matrix
3.  sum the one-hot rows over each window  ->  the composition matrix
4.  KMeans on the composition matrix       ->  a niche label per cell
```

Steps 1&ndash;3 are the function below. It is worth reading, because the parent course's
version of it is 25 lines longer for reasons that no longer apply to us.

> **What we dropped, and why.** The original `get_windows()` was written for multi-FOV CosMx
> data, where cells are split across dozens of separate fields of view that must never be
> neighbours of one another. It therefore carried a `tissue_group` groupby, an `exps` list of
> region names, `(start_time, idx, tissue_name, indices)` job tuples and an `np.array_split`
> chunking loop &mdash; all of it machinery for *keeping regions apart*. Our Crop is **one**
> continuous region, so all of that collapses to a single `NearestNeighbors(...).kneighbors()`
> call. The parent's names are kept in the comments so you can find your way back.

In [ ]:
def neighbourhood_composition(xy, labels, k):
    """Fraction of each cell type among the k nearest cells, for every cell.

    Returns a DataFrame of shape (n_cells, n_types); each row sums to 1.

    Parent-course lineage (`get_windows` + the `out_dict` / `windows` loops in
    3.2_neighborhood.ipynb): `onehot` is their `values`, `idx` is their `neighbors`,
    and the return value is their `windows[k]`. Their per-region bookkeeping
    (`tissue_group`, `exps`, `tissue_chunks`) is dropped -- we have one region.
    """
    onehot = pd.get_dummies(labels).astype(np.float32)     # (n_cells, n_types)
    nn = NearestNeighbors(n_neighbors=k).fit(xy)
    _, idx = nn.kneighbors(xy)                             # (n_cells, k), includes self
    counts = onehot.values[idx].sum(axis=1)                # (n_cells, n_types)
    return pd.DataFrame(counts / k, index=labels.index, columns=onehot.columns)


K_NEIGHBOURS = 20        # "nearby" = the 20 nearest cells, roughly a 60 µm radius here
N_NICHES = 8             # how finely we divide the tissue

t0 = time.time()
composition = neighbourhood_composition(xy, adata.obs["cell_type"], K_NEIGHBOURS)
print(f"composition matrix {composition.shape} in {time.time() - t0:.1f} s")
print()
print("the first three cells' neighbourhoods:")
print(composition.head(3).round(2).to_string())

Each row is one cell's surroundings expressed as 13 fractions that sum to 1. The first cell
above is 45% tumour, 25% fibroblast, 15% T cell; the third is 90% tumour and nothing else.
Those are the vectors we cluster.

**Why KMeans, and why `MiniBatchKMeans`?** We want hard, exhaustive, cheap groups: every cell
gets exactly one niche, and 16,006 &times; 13 is small. `MiniBatchKMeans` fits on random
subsets rather than the whole matrix each iteration &mdash; on this size it is a convenience
rather than a necessity, but it is what the parent course used and it scales to the millions
of cells you get from a full slide.

`n_clusters=8` is a *choice*. Not a discovery, not an optimum. Exercise 1 asks you to change
it and watch the story change with it.

In [ ]:
t0 = time.time()
kmeans = MiniBatchKMeans(n_clusters=N_NICHES, random_state=SEED, n_init=10)
adata.obs["niche"] = pd.Categorical(kmeans.fit_predict(composition.values))
print(f"MiniBatchKMeans in {time.time() - t0:.1f} s")
print()
print(adata.obs["niche"].value_counts().sort_index().to_string())

### 4.1 What is each niche made of?

A KMeans centroid *is* the average neighbourhood composition of its niche, so we can read the
niches straight off the model &mdash; no extra computation needed.

Raw fractions are hard to compare, because a niche that is 10% T cell means something very
different from a niche that is 10% tumour when the tissue is 44% tumour overall. So we plot
the **log2 fold change** of each niche's composition against the whole-Crop average: positive
(red) means enriched relative to the tissue, negative (blue) means depleted.

> **The `+ overall` in the formula is a pseudocount**, carried over from the parent course.
> Some niches contain literally zero cells of some type, and `log2(0)` is minus infinity,
> which no colour map can draw. Adding the tissue average before taking the ratio keeps every
> value finite. It also compresses the scale &mdash; the numbers on the colour bar are damped
> and should be read as *ranks and signs*, not as literal fold changes.

In [ ]:
centroids = pd.DataFrame(kmeans.cluster_centers_, columns=composition.columns,
                         index=[f"niche {i}" for i in range(N_NICHES)])
overall = composition.values.mean(axis=0)          # the Crop's average neighbourhood

# The parent course's cell 23, with its pseudocount, spelled out.
numerator = (centroids + overall).div((centroids + overall).sum(axis=1), axis=0)
log2fc = np.log2(numerator / overall)

g = sns.clustermap(log2fc, cmap="bwr", center=0, row_cluster=False,
                   figsize=(9.5, 4.8), linewidths=0.4,
                   cbar_kws={"label": "log2 fold change vs whole Crop"})
g.ax_heatmap.set_title("Niche composition, relative to the tissue average", pad=70)
plt.show()

### 4.2 Name them

The columns cluster into the compartments you would expect: tumour epithelial with
proliferating tumour; the immune populations together; endothelial with perivascular and
fibroblast. The rows are the eight niches, and each has an obvious character.

Now we give them names. **This is a judgement call, and it should look like one.** Rather than
typing eight strings against eight cluster ids &mdash; which silently breaks the moment KMeans
returns its clusters in a different order &mdash; we write the judgement down as a short,
readable rule table over the composition. Rules are tried in order; the first one that matches
wins.

Read the thresholds and disagree with them. That is the point: they are the part of this
analysis that came out of a person, not out of the data.

In [ ]:
IMMUNE_TYPES = ["T cell", "Macrophage", "Dendritic cell", "Plasma cell",
                "Mast", "Mixed (plasma+mast)"]


def name_niche(comp):
    # comp is one niche's mean neighbourhood composition (fractions summing to 1)
    tumour = comp["Tumour epithelial"] + comp["Proliferating tumour"]
    proliferating = comp["Proliferating tumour"] / max(tumour, 1e-9)   # share of tumour
    immune = comp[IMMUNE_TYPES].sum()
    vessel = comp["Endothelial"] + comp["Perivascular"]

    if tumour >= 0.90 and proliferating >= 0.10:
        return "Proliferating tumour core"      # solid tumour, actively cycling
    if tumour >= 0.90:
        return "Tumour core"                    # solid tumour, quiescent
    if tumour >= 0.55 and proliferating >= 0.25:
        return "Proliferating tumour front"     # tumour meeting stroma, cycling hard
    if tumour >= 0.55:
        return "Tumour boundary"                # tumour meeting stroma
    if comp["Myoepithelial"] >= 0.20:
        return "Myoepithelial border"           # the duct wall itself
    if immune >= 0.60:
        return "Immune infiltrate"              # stroma dominated by immune cells
    if comp["Fibroblast"] >= 0.30:
        return "Fibroblast stroma"              # collagenous stroma
    if vessel >= 0.15:
        return "Vascular stroma"                # stroma organised around vessels
    return "Mixed stroma"                       # nothing dominant


NICHE_ORDER = ["Tumour core", "Proliferating tumour core", "Tumour boundary",
               "Proliferating tumour front", "Myoepithelial border",
               "Vascular stroma", "Fibroblast stroma", "Immune infiltrate",
               "Mixed stroma"]

names = centroids.apply(name_niche, axis=1)
if names.nunique() < N_NICHES:                  # two niches given the same name
    seen = {}
    fixed = []
    for n in names:
        seen[n] = seen.get(n, 0) + 1
        fixed.append(n if seen[n] == 1 else f"{n} ({seen[n]})")
    names = pd.Series(fixed, index=names.index)
    print("NOTE: the rules gave a duplicate name; numbered suffixes were added.\n")

present = [n for n in NICHE_ORDER if n in set(names)] + \
          [n for n in names if n not in NICHE_ORDER]
adata.obs["niche_name"] = pd.Categorical(
    names.values[adata.obs["niche"].cat.codes.values], categories=present, ordered=True)

summary = pd.DataFrame({
    "name": names.values,
    "cells": adata.obs["niche"].value_counts().sort_index().values,
    # the three quantities the rules above actually test
    "%tumour": (100 * (centroids["Tumour epithelial"]
                       + centroids["Proliferating tumour"])).round(0).values,
    "%immune": (100 * centroids[IMMUNE_TYPES].sum(axis=1)).round(0).values,
    "%fibro": (100 * centroids["Fibroblast"]).round(0).values,
    "%vessel": (100 * (centroids["Endothelial"]
                       + centroids["Perivascular"])).round(0).values,
    "top three cell types": [
        ", ".join(f"{t} {100 * v:.0f}%" for t, v in row.sort_values(ascending=False).head(3).items())
        for _, row in centroids.iterrows()],
}, index=centroids.index)
print(summary.to_string())

> **Read the ordered rules literally.** `Fibroblast stroma` comes out with slightly *more*
> vessel content than `Vascular stroma` does. It is named for its fibroblasts anyway, because
> the fibroblast rule is tested first and "39% fibroblast" is the more distinctive fact about
> it. Ordered rules always behave like this, and it is exactly why the rules are written out
> above the table instead of hidden in a lookup dictionary &mdash; you can see which one fired
> and argue with the order. (Section 5.5 gives an independent check that `Vascular stroma`
> deserves its name: endothelial and perivascular cells really are closest to it.)

### 4.3 Put the niches back on the tissue

The clustermap said what the niches are *made of*. This says where they *are* &mdash; and it
is the figure that makes the method click, because the niches were built with **no knowledge
of position whatsoever**. KMeans saw 16,006 rows of 13 fractions, in no particular order. It
had no idea which cell was next to which.

And yet the labels come out spatially coherent: contiguous domains with sharp edges, not
confetti. That coherence is not an assumption we imposed. It is evidence that tissue really
is organised into recurring local arrangements.

In [ ]:
NICHE_COLOURS = {
    "Tumour core":                "#8b0000",   # dark red   -- solid tumour
    "Proliferating tumour core":  "#c1272d",   # red
    "Tumour boundary":            "#e8743b",   # orange     -- tumour meeting stroma
    "Proliferating tumour front": "#f5b041",   # light orange
    "Myoepithelial border":       "#d4a017",   # gold       -- the duct wall
    "Vascular stroma":            "#2e8b57",   # green      -- vessels
    "Fibroblast stroma":          "#8c6d3f",   # brown      -- collagen
    "Immune infiltrate":          "#1f6fb4",   # blue       -- immune
    "Mixed stroma":               "#9e9e9e",   # grey
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6.6))
for ct in CELL_TYPES:
    m = (adata.obs["cell_type"] == ct).values
    axes[0].scatter(xy[m, 0], xy[m, 1], s=1.6, c=PALETTE[ct], linewidths=0)
axes[0].set_title("what we started from: cell types")

for i, n in enumerate(adata.obs["niche_name"].cat.categories):
    m = (adata.obs["niche_name"] == n).values
    niche_id = int(names[names == n].index[0].split()[-1])
    axes[1].scatter(xy[m, 0], xy[m, 1], s=1.6, color=NICHE_COLOURS.get(n, "0.6"),
                    linewidths=0, label=f"{niche_id} · {n} ({m.sum():,})")
axes[1].set_title(f"what we built: {N_NICHES} niches (k = {K_NEIGHBOURS})")
axes[1].legend(markerscale=6, fontsize=7.5, bbox_to_anchor=(1.02, 1), loc="upper left")

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(HE_EXTENT[0], HE_EXTENT[1])
    ax.set_ylim(HE_EXTENT[2], HE_EXTENT[3])
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

Compare the two panels and the picture assembles itself:

* The solid red carcinoma masses on the left become **tumour core** (dark red), with a thin
  **tumour boundary** shell (orange) wrapped around each one and a **myoepithelial border**
  (gold) where a duct still has its wall.
* The scattered blue and cyan immune cells on the left, which look like noise, resolve into a
  connected **immune infiltrate** domain (blue) filling the stroma between the tumour masses.
* **Fibroblast stroma** (brown) and **vascular stroma** (green) split the rest of the
  interstitium between them.

The immune infiltrate is the largest single niche in this Crop and it is *everywhere between*
the tumour, and *almost nowhere inside* it. Section 3 told us that with one z-score. Now we
have it per cell, and can measure it.

> **What did not appear:** there is no niche of pure vasculature at `n_clusters = 8`.
> Endothelial and perivascular cells are enriched in the green niche but never dominate one,
> because vessels in this tissue are thin structures embedded in stroma rather than territories
> of their own. Push `n_clusters` higher and a vessel niche does separate out &mdash; which is
> Exercise 1, and a good illustration that the number of clusters decides which biology you
> are allowed to see.

### 4.4 A closer look: Voronoi tessellation

Dots lose their spatial extent. A **Voronoi tessellation** gives each cell the territory of
all points closer to it than to any other cell, so the tissue is tiled with no gaps and the
niche domains become solid regions with visible borders.

We draw it on the same 500 µm sub-crop as Section 3.1 &mdash; about 1,300 cells. The helper
loops over cells in Python and clips each polygon against the convex hull, so all 16,006
cells would be slow *and* would produce a figure whose cells are smaller than one pixel.

> `draw_voronoi_scatter` comes from `voronoi.py`, vendored unchanged from the parent course
> apart from three fixes &mdash; the most instructive being that the original called
> `vor.points.ptp()`, and **`ndarray.ptp()` was removed in NumPy 2.0**. Colab ships numpy
> 2.0.2, so the original code raises `AttributeError` on the first call. If you inherit
> analysis code written before 2024, grep it for `.ptp()`, `np.float_`, `np.NaN`, `np.in1d`
> and `.newbyteorder()` before you trust it.

In [ ]:
if not HAVE_VORONOI:
    print("voronoi.py was not available -- skipping this figure. Nothing else is affected.")
else:
    niche_categories = list(adata.obs["niche_name"].cat.categories)
    voronoi_palette = [NICHE_COLOURS.get(n, "0.6") for n in niche_categories]

    spot = pd.DataFrame({
        "x": xy[in_window, 0],
        "y": xy[in_window, 1],
        # draw_voronoi_scatter indexes the palette by this column, so it must be
        # integer codes into `voronoi_palette`, not strings.
        "niche_code": adata.obs["niche_name"].cat.codes.values[in_window],
    })

    t0 = time.time()
    areas = draw_voronoi_scatter(spot, [], voronoi_palette=voronoi_palette,
                                 X="x", Y="y", voronoi_hue="niche_code",
                                 figsize=(7.2, 7.2))
    handles = [plt.Line2D([], [], marker="s", ls="", markersize=8,
                          color=voronoi_palette[i], label=n)
               for i, n in enumerate(niche_categories)
               if (spot["niche_code"] == i).any()]
    plt.legend(handles=handles, fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.title(f"Voronoi tessellation of {len(spot):,} cells, coloured by niche")
    plt.show()
    print(f"{len(spot):,} cells tessellated in {time.time() - t0:.1f} s; "
          f"median cell territory {np.median(areas):.0f} µm²")

Now the niches are territories with borders, and you can see how tightly the tumour niches
are packed compared with the loose, vessel-punctuated stroma. You can also see the boundary
niches doing their job &mdash; a one-to-two-cell-thick band tracing the edge of each tumour
mass, which is precisely the structure a per-pair z-score cannot represent.

### 4.5 `k` is the spatial-scale knob

`k = 20` was a choice, and it is the single most consequential one in the whole method.

* **Small `k`** &mdash; the window is a handful of touching cells. Niches become sensitive to
  fine structure, and also to segmentation noise: one misassigned cell moves the vector a lot.
* **Large `k`** &mdash; the window spans many cell diameters. Niches become smooth regional
  domains, and genuinely thin structures (a two-cell-thick boundary, a capillary) are averaged
  out of existence.

Neither is right. `k` sets *the scale at which you are asking the question*, so it should be
chosen from the biology: if you care about which cells touch a tumour edge, small; if you care
about which region of the slide is inflamed, large.

The next cell re-runs everything at `k = 5`, `20` and `50`. It takes about half a minute.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.8))
handles = {}
for ax, k in zip(axes, [5, 20, 50]):
    comp_k = neighbourhood_composition(xy, adata.obs["cell_type"], k)
    km_k = MiniBatchKMeans(n_clusters=N_NICHES, random_state=SEED, n_init=10).fit(comp_k.values)
    names_k = pd.DataFrame(km_k.cluster_centers_,
                           columns=comp_k.columns).apply(name_niche, axis=1).values
    labels_k = names_k[km_k.labels_]

    for n in sorted(set(labels_k)):
        m = labels_k == n
        h = ax.scatter(xy[m, 0], xy[m, 1], s=2.0, color=NICHE_COLOURS.get(n, "0.6"),
                       linewidths=0)
        handles.setdefault(n, h)
    counts = pd.Series(labels_k).value_counts()
    print(f"k = {k}:  {counts.size} distinct names out of {N_NICHES} clusters")
    print("   " + counts.to_string().replace("\n", "\n   "))
    ax.set_title(f"k = {k}   ({counts.size} distinct niche names)")
    ax.set_aspect("equal")
    ax.set_xlim(HE_EXTENT[0], HE_EXTENT[1])
    ax.set_ylim(HE_EXTENT[2], HE_EXTENT[3])
    ax.set_xticks([])
    ax.set_yticks([])

order = [n for n in NICHE_ORDER if n in handles]
fig.legend([handles[n] for n in order], order, markerscale=7, fontsize=8,
           bbox_to_anchor=(1.005, 0.92), loc="upper left")
plt.suptitle("The same 8 clusters, three neighbourhood sizes", y=1.02)
plt.tight_layout()
plt.show()

Three things to take from that figure and its printed counts.

**At `k = 5` the map is grainy and the niches are coarse.** Single cells flip between niches,
because a five-cell window is dominated by whatever happens to be immediately adjacent. Worse,
*two of the eight clusters end up with the same name*: with only five neighbours, almost every
window inside a tumour mass is 100% tumour, so KMeans spends several clusters splitting hairs
between near-identical all-tumour vectors and the tissue collapses to six distinguishable
kinds of place, with a single 6,000-cell `Tumour core`.

**At `k = 50` the thin structures dissolve.** No 50-cell window is ≥ 90% tumour any more, so
`Tumour core` shrinks and `Tumour boundary` balloons to nearly a third of the Crop; both
proliferating niches disappear, because averaging over 50 cells dilutes the proliferating
fraction below the threshold; and the catch-all `Mixed stroma` appears, holding 2,000 cells
whose neighbourhoods are now too blended for any rule to fire. The picture is smooth and
regional. That is not worse &mdash; it is a different question, answered well.

**The names are a readout of the scale.** The naming rules are applied to whatever KMeans
returns, so a `k` that stops resolving a proliferating core simply stops producing that name.
Only `k = 20` yields all eight distinct, which is a reasonable &mdash; but not sacred &mdash;
reason to have picked it.

> **In your own work, say what `k` was and why.** "Niche" is not a property of a tissue; it
> is a property of a tissue *at a stated spatial scale*. A niche paper that does not report
> its neighbourhood size has not reported its method.

---

## 5. sopa: niches as measurable objects

So far a niche has been a colour on a scatter plot. `sopa` turns it into a **shape** &mdash; a
polygon with an area, a perimeter and a roundness &mdash; and into a **distance** &mdash; how
many cells you have to walk through to get from any given cell to the nearest cell of a given
niche.

That second one is the payoff. "This tumour is immune-excluded" becomes a number per cell.

Four functions do all of it:

| Function | Question it answers |
| --- | --- |
| `spatial_neighbors(adata, radius)` | build the graph everything else walks on |
| `vectorize_niches(adata, key)` | what shape is each niche, and how many pieces is it in? |
| `niches_geometry_stats(adata, key)` | how big / round / far apart are the niches? |
| `cells_to_groups(adata, key)` | how far is **each cell** from each niche? |
| `mean_distance(adata, a, b)` | how far is each cell type from each niche, on average? |

> **`sopa`'s API moves.** These are the names in **sopa 2.2.x**. Earlier versions had
> `geometrize_niches` where 2.2 has `vectorize_niches`, and tutorials written against
> 2.0 will fail here. The next cell checks every name this section uses **before** running
> any of them, so that a version mismatch produces one clear sentence rather than a traceback
> six cells later.

In [ ]:
# Fail loudly and early if this sopa does not have what the section needs.
REQUIRED = ["spatial_neighbors", "vectorize_niches", "niches_geometry_stats",
            "cells_to_groups", "mean_distance"]
missing = [f for f in REQUIRED if not hasattr(sopa.spatial, f)]

print(f"sopa {sopa.__version__}")
for f in REQUIRED:
    print(f"  sopa.spatial.{f:<22} {'present' if hasattr(sopa.spatial, f) else 'MISSING'}")

if missing:
    raise ImportError(
        f"This section needs sopa 2.2.x. Installed: {sopa.__version__}, which is missing "
        f"{', '.join(missing)}. In sopa < 2.1 `vectorize_niches` was called "
        f"`geometrize_niches`. Fix with:  %pip install -q 'sopa==2.2.10'  then "
        f"Runtime -> Restart session."
    )
print("\nall required sopa functions present")

### 5.1 A graph you can measure distances on

`sopa.spatial.spatial_neighbors` is squidpy's Delaunay builder with one addition: a
**required** `radius` argument that prunes edges outside a distance interval.

That prune is not cosmetic. Section 3 found Delaunay edges up to 1,700 µm long, spanning
empty duct lumens and the convex hull. A hop count along a graph containing those shortcuts
would say a cell is "2 cells away" from a niche on the other side of the Crop. `radius=[0, 50]`
keeps edges between 0 and 50 µm &mdash; a couple of cell diameters &mdash; so a hop is a real
step between touching cells.

This call **replaces** the graph in `adata.obsp` that Section 3 built. That is fine: we are
done with the enrichment test, and its results are already stored in `adata.uns`.

In [ ]:
t0 = time.time()
sopa.spatial.spatial_neighbors(adata, radius=[0, 50])      # radius is positional-required
print(f"pruned Delaunay graph in {time.time() - t0:.1f} s")

A_sopa = adata.obsp["spatial_connectivities"]
print(f"  {A_sopa.nnz // 2:,} edges "
      f"(was {A.nnz // 2:,} before pruning at 50 µm)")
print(f"  mean {A_sopa.nnz / adata.n_obs:.1f} neighbours per cell")

### 5.2 Niches as polygons

`vectorize_niches` walks the graph, finds the connected groups of same-niche cells, and turns
each group into a polygon. A niche is usually **several** polygons &mdash; sopa calls them
*components* &mdash; because the same kind of place recurs in different spots. That is the
"kind of place, not a place" idea from Section 2, made concrete.

Two parameters shape the result:

* `buffer` &mdash; how far each component is expanded before its cells are merged into one
  shape. `"auto"` uses three times the mean distance between neighbouring cells. Bigger buffer
  = fewer, fatter, more merged components.
* `perc_area_th=0.05` &mdash; drop components smaller than 5% of the biggest component of the
  same niche, so a single stray cell does not become a "niche".

In [ ]:
t0 = time.time()
niche_shapes = sopa.spatial.vectorize_niches(adata, "niche_name",
                                             buffer="auto", perc_area_th=0.05)
print(f"vectorize_niches in {time.time() - t0:.1f} s")
print(f"{len(niche_shapes)} components across "
      f"{niche_shapes['niche_name'].nunique()} niches")
print()
print(niche_shapes.drop(columns="geometry").head().round(1).to_string())

fig, ax = plt.subplots(figsize=(7.6, 7.2))
for n, part in niche_shapes.groupby("niche_name", observed=True):
    part.plot(ax=ax, color=NICHE_COLOURS.get(n, "0.6"), edgecolor="white",
              linewidth=0.6, alpha=0.85, label=n)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm)")
ax.set_title('Niches as polygons (buffer="auto")')
ax.legend(fontsize=7.5, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 5.3 Niche geometry, as a table

`niches_geometry_stats` aggregates those polygons into one row per niche: how many components
it breaks into, the mean perimeter (`length`) and area of a component, its `roundness`
(4&pi;&middot;area / length², so 1 is a perfect circle and 0 is an infinitely ragged sliver),
and the distance from each niche to each other niche.

The `aggregation="min"` argument says: for the niche-to-niche distances, report the *closest*
approach between any component of A and any component of B. `"mean"` is also available and
answers a different question &mdash; typical separation rather than closest contact.

In [ ]:
t0 = time.time()
geom = sopa.spatial.niches_geometry_stats(adata, "niche_name", aggregation="min")
print(f"niches_geometry_stats in {time.time() - t0:.1f} s\n")

shape_cols = ["n_components", "length", "area", "roundness"]
print("shape of each niche:")
print(geom[shape_cols].round(1).to_string())
print()
print("closest approach between niches, µm:")
dist_cols = [c for c in geom.columns if c.startswith("min_distance_to_niche_")]
approach = geom[dist_cols].copy()
approach.columns = [c.replace("min_distance_to_niche_", "") for c in dist_cols]
print(approach.round(0).to_string())

Read the shape table as morphology, because that is what it is.

**`n_components` separates continuous niches from scattered ones.** The tumour boundary niche
breaks into the most pieces &mdash; of course it does, there is one boundary per tumour mass.
The immune infiltrate breaks into the fewest and has the largest mean area: it is one big
connected stromal territory, not a set of isolated pockets. That single number is the
difference between "diffusely infiltrated" and "a few immune aggregates", and it is the sort
of thing you would want to compare between a responder and a non-responder.

**`roundness` is low for everything (~0.3&ndash;0.4).** These are irregular, interdigitating
territories, not blobs. A tumour core component approaching roundness 1 would mean a smooth
pushing margin; ragged components mean an infiltrative one.

> **These numbers depend on `buffer`, and you must say which you used.** Buffer decides what
> counts as one connected piece. The next cell re-runs the same call with a fixed 30 µm buffer
> so you can see how much moves.

In [ ]:
alt = sopa.spatial.vectorize_niches(adata, "niche_name", buffer=30, perc_area_th=0.05)

comparison = pd.DataFrame({
    'components, buffer="auto"': niche_shapes.groupby("niche_name", observed=True).size(),
    "components, buffer=30 µm": alt.groupby("niche_name", observed=True).size(),
    'mean area, buffer="auto"': niche_shapes.groupby("niche_name", observed=True)["area"].mean(),
    "mean area, buffer=30 µm": alt.groupby("niche_name", observed=True)["area"].mean(),
}).round(0)
print(comparison.to_string())
print()
print(f'total components: buffer="auto" -> {len(niche_shapes)}, '
      f"buffer=30 µm -> {len(alt)}")

The component counts move substantially, and so do the areas: a smaller buffer keeps
neighbouring patches separate, a larger one glues them together. **The ranking between niches
is stable; the absolute numbers are not.** Compare niches within one buffer setting, never a
niche in one analysis against a niche in another that used a different buffer.

### 5.4 The payoff: how far is every cell from the immune niche?

`cells_to_groups` walks the pruned graph from every cell outward and records, for each niche,
how many hops it takes to reach the nearest cell of that niche. Zero means the cell is *in*
that niche. Ten means you pass ten cells to get there.

Hops, not micrometres, and that is deliberate: a hop is a *cellular* distance. Ten hops
through dense tumour is a shorter physical distance than ten hops through loose stroma, but it
is the same number of cell-cell interfaces to cross &mdash; which is what a migrating
lymphocyte actually experiences.

In [ ]:
t0 = time.time()
sopa.spatial.cells_to_groups(adata, "niche_name", key_added_prefix="distance_to_")
print(f"cells_to_groups in {time.time() - t0:.1f} s")

hops = adata.obsm["distance_to_niche_name"]
print(f"{hops.shape[0]:,} cells x {hops.shape[1]} niches")
print()
print("hop distance to each niche, over all cells:")
print(hops.describe().loc[["mean", "50%", "max"]].round(1).to_string())
if hops.isna().any().any():
    print(f"\n({int(hops.isna().any(axis=1).sum())} cell(s) unreachable after the 50 µm "
          "prune -- isolated in the graph, shown in grey below)")

In [ ]:
IMMUNE_NICHE = "Immune infiltrate"
d_immune = hops[IMMUNE_NICHE].values

fig, axes = plt.subplots(1, 2, figsize=(14, 6.4))

finite = np.isfinite(d_immune)
axes[0].scatter(xy[~finite, 0], xy[~finite, 1], s=2, c="0.85", linewidths=0)
sc = axes[0].scatter(xy[finite, 0], xy[finite, 1], c=d_immune[finite], s=2.2,
                     cmap="magma_r", vmin=0, vmax=15, linewidths=0)
plt.colorbar(sc, ax=axes[0], shrink=0.75, label=f"hops to the nearest {IMMUNE_NICHE} cell")
axes[0].set_title(f"Every cell's distance to the {IMMUNE_NICHE.lower()} niche")

by_type = (pd.Series(d_immune, index=adata.obs_names)
           .groupby(adata.obs["cell_type"], observed=True).median()
           .sort_values())
axes[1].barh(by_type.index, by_type.values,
             color=[PALETTE[c] for c in by_type.index])
axes[1].set_xlabel(f"median hops to the {IMMUNE_NICHE.lower()} niche")
axes[1].set_title("...summarised by cell type")
sns.despine(ax=axes[1])

for ax in [axes[0]]:
    ax.set_aspect("equal")
    ax.set_xlim(HE_EXTENT[0], HE_EXTENT[1])
    ax.set_ylim(HE_EXTENT[2], HE_EXTENT[3])
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

**This is the figure to take away.** On the left, **pale means "in, or right beside, the
immune niche"** and **dark means "a long walk from any immune cell"**. The carcinoma masses
are the dark islands, each ringed by a pale shoreline: the immune compartment presses against
the tumour everywhere and penetrates it almost nowhere. The interiors of the largest masses
are the darkest regions in the Crop, at 15 or more cell-to-cell steps from the nearest immune
niche cell.

On the right, the same information per cell type. Every immune population sits within a hop
or two. Fibroblasts, endothelium and perivascular cells are close, because that is where
immune cells are. Tumour epithelium is the furthest of anything in the tissue.

That ordering *is* the immune-exclusion phenotype, measured. And unlike the z-score in
Section 3, it is per cell, so you could take the tumour cells in the top decile of this
distance and ask what is different about their transcriptome.

### 5.5 Cell types against niches, in one table

`mean_distance` does the same walk but reports a mean per (cell type, niche) pair.

In [ ]:
t0 = time.time()
md_table = sopa.spatial.mean_distance(adata, "cell_type", "niche_name")
print(f"mean_distance in {time.time() - t0:.1f} s")

md_table = md_table[[n for n in adata.obs["niche_name"].cat.categories
                     if n in md_table.columns]]

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(md_table, cmap="magma_r", annot=True, fmt=".1f", linewidths=0.4,
            cbar_kws={"label": "mean hops"}, ax=ax)
ax.set_title("Mean hop distance from each cell type to each niche")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

Read the extremes along the diagonal of the story:

* **T cell &rarr; immune infiltrate: under one hop.** T cells are in that niche, by
  construction. **T cell &rarr; tumour core: eight hops.** Eight cells of tumour to push
  through.
* **Tumour epithelial &rarr; tumour core: under two hops. &rarr; immune infiltrate: nine.**
  The mirror image. The two compartments are each other's furthest neighbours.
* **Endothelial and perivascular &rarr; vascular stroma: ~1.4 hops**, the closest pairing in
  the table after self-association. The vasculature niche is doing what its name says, which
  is a useful sanity check on the naming rules from Section 4.2.
* **Dendritic cells are closer to the tumour niches than T cells are.** Antigen-presenting
  cells get nearer the tumour than the effectors do &mdash; a difference you could not see at
  all in the Section 3 heatmap, and one worth arguing about over coffee.

### 5.6 One more, if you have `netgraph`

`sopa.spatial.prepare_network` turns everything above into a graph: cell types and niches
are both nodes, and the edge between any two is `1 / mean hop distance` &mdash; so a thick edge
means "these two are close in tissue".

Two practical notes. `netgraph` is **not** a `sopa` dependency; it is named explicitly in
this Tutorial's install line, and the cell still checks before importing. And `sopa` hands
back a square *adjacency table*, which `netgraph` will not take directly &mdash; convert it with
`networkx.from_pandas_adjacency` first. Every pair has some distance, so the raw graph is
fully connected (21 nodes, 210 edges) and unreadable; we keep only the strongest edges.

In [ ]:
try:
    import netgraph                                     # noqa: F401
    import networkx as nx
    have_netgraph = True
except ImportError:
    have_netgraph = False

if not have_netgraph:
    print("netgraph is not installed, so prepare_network is skipped.")
    print("It is a plotting convenience, not part of the analysis:")
    print("everything it draws is already in the mean_distance table above.")
    print("To try it:  %pip install -q netgraph   then re-run this cell.")
else:
    weights, node_colours, node_sizes, node_shapes = sopa.spatial.prepare_network(
        adata, cell_type_key="cell_type", niche_key="niche_name")

    # sopa returns a square adjacency table; netgraph wants a graph object.
    G = nx.from_pandas_adjacency(weights)

    # Every pair has a finite distance, so G is complete and would draw as a hairball.
    # Keep the strongest quarter of edges -- the ones that carry the story.
    w = np.array([d["weight"] for _, _, d in G.edges(data=True)])
    cutoff = np.quantile(w[w > 0], 0.75)
    G.remove_edges_from([(u, v) for u, v, d in G.edges(data=True) if d["weight"] < cutoff])
    print(f"showing {G.number_of_edges()} strongest edges of {len(weights) * (len(weights) - 1) // 2}"
          f"  (weight >= {cutoff:.2f})")

    plt.figure(figsize=(9, 9))
    netgraph.Graph(
        G,
        node_color=node_colours, node_size=node_sizes, node_shape=node_shapes,
        edge_width={(u, v): 6 * d["weight"] for u, v, d in G.edges(data=True)},
        node_labels=True, node_label_fontdict={"size": 8}, edge_alpha=0.3,
    )
    plt.title("Cell types (circles) and niches (hexagons), linked by proximity")
    plt.show()


---

## 6. Exercises

Each of these is a small edit to a cell above, followed by re-running from that cell down.

**1. How many niches?** `N_NICHES = 8` was a choice. Re-run Section 4 with `4` and with `15`.
At 4 you should get something close to tumour / boundary / stroma / immune. At 15 the naming
rules will start returning duplicates with numbered suffixes &mdash; look at the clustermap
and decide whether the extra clusters are biology (a vessel niche finally separating out?) or
just subdivided noise. What would convince you either way?

**2. Cluster on the vendor's labels instead.** `adata.obs` also carries `graphclust` (34
unsupervised clusters) and `kmeans_10`, straight from the 10x pipeline, with no cell-type
interpretation applied. Pass `adata.obs["graphclust"]` to `neighbourhood_composition` instead
of `cell_type`. You now have niches built from clusters nobody has named. Do they land in the
same places? If they do, that tells you the niche structure is robust to how the cells were
labelled &mdash; a much stronger claim than either analysis alone.

**3. Change the spatial scale properly.** Section 4.5 varied `k` with `N_NICHES` fixed. Try
varying both: `k = 10, N_NICHES = 12` against `k = 40, N_NICHES = 5`. Which pairing gives
niches you would be willing to defend in a figure legend?

**4. Go and find the B cells (the interesting one).** Section 1.4 showed that no B cell
cluster was resolved. Score every cell for the four B-cell genes &mdash;
`score = X[:, [MS4A1, CD79A, CD79B, BANK1]].sum(axis=1)` &mdash; take the top 1% of cells, and
plot where they sit. Then cross-tabulate them against `niche_name`. Are they scattered
uniformly, or concentrated in one niche? A concentration in the immune infiltrate niche,
especially near vessels, is what a **tertiary lymphoid structure** looks like &mdash; and
finding one this way, without ever having a B cell cluster, is a good demonstration of why
niches are worth computing.

**5. Who is furthest from the tumour?** Section 5.4 plotted distance to the immune niche.
Plot distance to `"Tumour core"` instead. Which immune population gets closest, and does that
match the dendritic-cell observation in 5.5?

**6. Break it on purpose.** Shuffle `adata.obs["cell_type"]` with
`np.random.permutation`, keeping the coordinates fixed, and re-run Sections 3 and 4. The
enrichment heatmap should go flat and the niches should become spatial confetti. If they do
not, something in your pipeline is finding structure that is not there &mdash; which is the
single most useful negative control in spatial analysis.

---

## Further reading

**The tool this Tutorial replaced.** [`hoodscanR`](https://bioconductor.org/packages/hoodscanR/)
(Bioconductor, R) is what the parent course uses for this analysis, and it is worth your time
if you work in R. Instead of hard KMeans it computes, for every cell, a *probability*
distribution over neighbourhoods, plus an entropy per cell that measures how mixed its
surroundings are &mdash; a boundary cell scores high, a cell deep in a tumour mass scores low.
That per-cell "how mixed am I" number is genuinely nicer than a hard label, and it is the main
thing you give up by staying in Python.

**Methods that go further than what we built.**

* [**CellCharter**](https://www.nature.com/articles/s41588-023-01588-4) (Nature Genetics, 2023)
  &mdash; the same neighbourhood idea, but it aggregates over several graph hops, uses a
  variational autoencoder to handle batch effects across samples, and fits a Gaussian mixture
  instead of KMeans, so niche count can be chosen by model selection rather than by hand.
* [**UTAG**](https://www.nature.com/articles/s41592-022-01657-2) (Nature Methods, 2022) &mdash;
  multiplies the *expression* matrix by the adjacency matrix before clustering, so niches are
  built from transcriptional state directly rather than from cell type labels. Useful when your
  cell typing is the weak link &mdash; which, given Section 1.4, it often is.
* [**Banksy**](https://www.nature.com/articles/s41588-024-01664-3) (Nature Genetics, 2024)
  &mdash; augments each cell's own expression with its neighbourhood average, and gets both
  cell types and niches out of one clustering, tuned by a single mixing parameter.
* [**squidpy**](https://squidpy.readthedocs.io/) and [**sopa**](https://gustaveroussy.github.io/sopa/)
  documentation &mdash; both have more spatial statistics than we touched: Ripley's K, Moran's
  I for spatially variable genes, ligand-receptor analysis restricted to neighbouring cells
  &mdash; and `squidpy.gr.ligrec` is the non-spatial version of what Tutorial 2b does next.

### Where this Tutorial sits

Tutorial 1 read the data and put it on screen. This Tutorial asked which cells sit next to
which, and turned the answer into per-cell labels and measurable objects. Tutorial 2b takes
the same 16,006 cells and asks what might be passing between them &mdash; adjacency is not
conversation, and a ligand-receptor test is how you look for the difference. Tutorial 3 asks what
you could have inferred from the H&E image alone &mdash; and finds, with an honesty that
rhymes with Section 1.4, that morphology predicts the stromal and epithelial compartments well
and the immune ones badly.

The through-line across all three: **every one of these methods has a boundary, and the
boundary usually lands on the immune compartment.** Knowing where it lands is the difference
between using these tools and being used by them.